# 13 — MCP：讓工具不綁死在某個框架上

**這份要學什麼**
- MCP 協定的概念：工具（server）跟呼叫工具的一方（client）分離，不綁死在某個框架上
- 本機 stdio 連線 vs 遠端 `streamable_http` 連線，差在設定方式而已

> 不需要 API key。這份 notebook 會啟動一個**真正的本機 MCP server**（用 stdio 子行程溝通，
> 完全不碰網路），示範怎麼把它的工具接進我們在 `05` 學過的 `StateGraph + ToolNode` 流程。

## MCP 是什麼：一套「萬用電源插座規格」

以前每個工具（查天氣、查股價、查公司內部系統……）都要自己寫一套「怎麼跟 agent 溝通」的
程式碼，換一個 agent 框架、換一個工具，就要重寫一次串接邏輯。

**MCP（Model Context Protocol，模型情境協定）** 是 Anthropic 提出的公開協定，作用像統一
插座規格：只要工具（server 端）跟呼叫工具的一方（client 端）都遵守同一份「怎麼描述工具、
怎麼呼叫」的規格，不管換成哪個 MCP server、哪個支援 MCP 的 agent 框架（LangGraph、
Claude Desktop、別人的 IDE……），都能直接插上去用，不用重寫任何串接程式碼。

## 這份 notebook 要做的事
啟動 `notebooks/_mcp_server.py`（一個提供 `get_weather` / `add` / `check_service_status` /
`lookup_runbook` 四個工具的本機 MCP server），用 `langchain-mcp-adapters` 把它的工具接進
`05` 學過的 `StateGraph + ToolNode` 流程，證明「MCP 工具」跟「本地 `@tool` 工具」對
`ToolNode` 來說完全沒有差別。`16` 的 capstone 會實際用到 `check_service_status` 跟
`lookup_runbook` 這兩個工具。

## Server 端：`_mcp_server.py` —— 廚房

把「MCP server」想成餐廳的廚房：負責真正把菜做出來（執行工具邏輯），但不管誰來點餐、
點餐單長什麼樣。

這個 repo 裡的 `notebooks/_mcp_server.py` 就是一個最小的 MCP server，用官方 `mcp` SDK 的
`FastMCP` 寫成：

```python
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("demo-tools")

@mcp.tool()
def get_weather(city: str) -> str:
    """Get the current weather for a city."""
    return f"{city}: sunny, 28C"

if __name__ == "__main__":
    mcp.run(transport="stdio")
```

`@mcp.tool()` 跟 LangChain 的 `@tool`（`05` 用過的那個）做的事幾乎一樣——把一個 Python
函式的簽名跟 docstring 轉成工具的說明書（schema），差別只在這裡輸出的格式是 MCP 的標準
格式，不是 LangChain 專屬的。這個檔案可以獨立執行（`python _mcp_server.py`），透過標準
輸入輸出（stdin/stdout）跟任何 MCP client 溝通——這就是下一段要說的 **stdio transport**。

## Client 端：`langchain-mcp-adapters` —— 外場服務生

如果 server 是廚房，**client** 就是外場服務生：服務生不用會做菜，只要用同一份「點餐單
格式」跟廚房溝通，把客人的需求轉成點餐單、把出好的菜端給客人。

官方的 `langchain-mcp-adapters` 套件就是這個服務生的角色，負責「連上 MCP server、把它的
工具轉成 LangChain 認得的 `BaseTool`」。`MultiServerMCPClient` 的設定用一個 dict，key 是
你自己取的 server 名稱，value 是連線方式——這裡用 `"stdio"`：直接把 server 當子行程啟動，
用標準輸入輸出溝通（**stdio transport**），全程不需要網路，也不用開任何 port。

三方關係長這樣：

```
Agent (StateGraph + ToolNode, 05)
   │  呼叫 BaseTool                  ▲ 收到 ToolMessage
   ▼                                 │
MultiServerMCPClient  (MCP client / 外場服務生)
   │  stdio：把 server 當子行程啟動   ▲
   │  用標準輸入輸出溝通              │
   ▼                                 │
_mcp_server.py  (MCP server / 廚房，工具真正執行的地方)
```

agent 完全不需要知道工具是本地寫的還是從 MCP server 來的——中間都被 client 轉成同一種
`BaseTool` 格式。

In [1]:
import sys

from langchain_mcp_adapters.client import MultiServerMCPClient

client = MultiServerMCPClient(
    {
        "demo": {
            "transport": "stdio",
            "command": sys.executable,  # 用同一個 venv 的 python 啟動 server 子行程
            "args": ["_mcp_server.py"],
        }
    }
)

mcp_tools = await client.get_tools()
print([t.name for t in mcp_tools])
print([t.description for t in mcp_tools])

['get_weather', 'add', 'check_service_status', 'lookup_runbook']
['Get the current weather for a city.', 'Add two numbers.', 'Check whether an internal service is healthy.', 'Look up an internal runbook entry for a known issue.']


`get_tools()` 是 async 的（因為要跟子行程溝通），Jupyter 的 cell 可以直接 `await`，不用另外
包 `asyncio.run()`。回傳的 `mcp_tools` 是一般的 `BaseTool` 物件列表——跟 `05` 裡
`@tool` 定義出來的工具是**同一個型別**。這才是重點：對 `ToolNode` 來說，工具是從 MCP
server 來的、還是本地 `@tool` 寫的，完全沒有差別。

## 一個小差異：MCP 工具的回傳值是「內容區塊」，不是純字串

MCP 規定工具結果要包成一組「內容區塊」（可以是文字、圖片……不只是字串），有點像寄信一定要
裝進信封，不能直接把紙條塞過去。所以直接 `.ainvoke()` 一個 MCP 工具，拿到的是
`[{"type": "text", "text": "..."}]` 這種列表，不是像 `05` 裡 `@tool` 那樣直接回傳字串。
`ToolNode` 知道怎麼拆開這個信封（下一個 cell 會示範），但如果你自己直接呼叫 `.ainvoke()`，
要記得這個形狀上的差異。

In [2]:
weather_tool = next(t for t in mcp_tools if t.name == "get_weather")
print(await weather_tool.ainvoke({"city": "Taipei"}))

[{'type': 'text', 'text': 'Taipei: sunny, 28C', 'id': 'lc_4adbb260-164e-4fee-9309-a94f21ce37e9'}]


## 接進 `05` 用過的 StateGraph + ToolNode 流程

把 `mcp_tools` 直接丟給 `ToolNode`，跟 `05` 的寫法一字不差——一樣用 `scripted_model` 走
離線劇本，證明 MCP 工具可以無縫替換掉本地 `@tool`。

In [ ]:
sys.path.insert(0, ".")
from _llm import get_llm, has_api_key, scripted_model

from langchain_core.messages import AIMessage, HumanMessage
from langgraph.graph import MessagesState, START, StateGraph
from langgraph.prebuilt import ToolNode, tools_condition

from _graph_viz import show_graph

model = scripted_model(
    [
        AIMessage(content="", tool_calls=[{"name": "get_weather", "args": {"city": "台北"}, "id": "c1"}]),
        AIMessage(content="台北目前是晴天，28 度。"),
    ]
).bind_tools(mcp_tools)


def call_model(state: MessagesState) -> dict:
    return {"messages": [model.invoke(state["messages"])]}


builder = StateGraph(MessagesState)
builder.add_node("agent", call_model)
builder.add_node("tools", ToolNode(mcp_tools))
builder.add_edge(START, "agent")
builder.add_conditional_edges("agent", tools_condition)
builder.add_edge("tools", "agent")
agent = builder.compile()

show_graph(agent)

result = await agent.ainvoke({"messages": [HumanMessage("台北天氣如何？")]})
for m in result["messages"]:
    print(f"{type(m).__name__:14} {m.content!r}")

注意 `ToolMessage` 的 `content` 是 MCP 的內容區塊列表，模型（這裡是劇本）看到的是這個
結構化格式，不影響它照樣讀出裡面的文字繼續對話。

## 如果你有 API key：接真模型 + 真的 MCP 工具
同一組 `mcp_tools`，換成 `get_llm()`，模型會自己決定要不要呼叫工具。

In [4]:
if has_api_key():
    from langchain.agents import create_agent

    real_agent = create_agent(model=get_llm(), tools=mcp_tools)
    real_result = await real_agent.ainvoke({"messages": [HumanMessage("台北天氣如何？")]})
    for m in real_result["messages"]:
        print(f"{type(m).__name__:14} {m.content!r}")
else:
    print("尚未設定 OPENAI_API_KEY，跳過真模型呼叫（上面的劇本版本已經展示了完整流程）。")

HumanMessage   '台北天氣如何？'
AIMessage      ''
ToolMessage    [{'type': 'text', 'text': '台北: sunny, 28C', 'id': 'lc_c07a2724-0fde-460c-8c42-030149847d5b'}]
AIMessage      '台北目前晴朗，氣溫約 **28°C**。'


## 廚房不一定在隔壁：遠端 MCP（`streamable_http` / `sse`）

前面的例子裡，廚房（server）跟外場（client）在同一間店裡，用 `"stdio"`（標準輸入輸出）
當傳菜口，直接把子行程啟動起來遞菜。但真實世界的 MCP server 常常是「中央廚房」——
架在別的機器上，透過網路提供服務（就像影片提到的高德地圖 MCP，是別人架好的遠端服務，
不是你自己啟動的子行程）。這種情況換成 `"streamable_http"`（或舊一點的 `"sse"`）
連線方式，設定從「怎麼啟動子行程」（`command` / `args`）變成「打去哪個網址」（`url`）。

下面把同一個 `_mcp_server.py` 用 HTTP 的方式在本機啟動（`127.0.0.1`，一樣不碰真的
網路），示範連線設定差在哪裡——除了怎麼連上去，其他程式碼（`get_tools()`、丟進
`ToolNode`）完全一樣。

In [5]:
import socket
import subprocess
import time

# 用 HTTP 方式在本機啟動同一個 MCP server（正式環境這裡會是別人已經架好的遠端網址）
# stdout/stderr 導去 DEVNULL：uvicorn 的 log 繼承到 Jupyter 的輸出管道會卡住，這裡用不到那些 log
server_process = subprocess.Popen(
    [
        sys.executable,
        "-c",
        'import sys; sys.path.insert(0, "."); from _mcp_server import mcp; '
        'mcp.settings.port = 8791; mcp.run(transport="streamable-http")',
    ],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)

for _ in range(50):  # 用 poll port 取代固定 sleep，避免 server 還沒起來就急著連
    try:
        with socket.create_connection(("127.0.0.1", 8791), timeout=0.2):
            break
    except OSError:
        time.sleep(0.2)

try:
    http_client = MultiServerMCPClient(
        {
            "demo": {
                "transport": "streamable_http",  # 差別只在這裡：不是 stdio，是網址
                "url": "http://127.0.0.1:8791/mcp",
            }
        }
    )
    http_tools = await http_client.get_tools()
    print([t.name for t in http_tools])
    print(await http_tools[0].ainvoke({"city": "Taipei"}))
finally:
    server_process.terminate()
    server_process.wait(timeout=5)

['get_weather', 'add', 'check_service_status', 'lookup_runbook']
[{'type': 'text', 'text': 'Taipei: sunny, 28C', 'id': 'lc_63e21625-3db9-4e9f-b532-c3d63af463cd'}]


## 小結
- MCP 是協定，不是框架——server 端可以用任何語言、跑在任何地方；`langchain-mcp-adapters`
  只是把 MCP 工具轉成 LangChain 認得的 `BaseTool`
- 轉換完之後，`05` 學過的 `ToolNode` / `tools_condition` / `bind_tools` 完全不用改，
  MCP 工具跟本地 `@tool` 工具在 graph 裡是同一等公民，不管 server 是本機 stdio 還是
  遠端 `streamable_http`
- 唯一要注意的差異：MCP 工具回傳的是內容區塊列表，不是純字串

`16_capstone_it_ticket_agent.ipynb` 會用這個 server 裡的 `check_service_status` /
`lookup_runbook` 工具，跟前面所有章節的技巧組成一個完整的實際應用場景。